In [5]:
import pandas as pd
import numpy as np
import nibabel as nib
import glob
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from monai.networks.nets import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

cuda


In [6]:
df = pd.read_excel("/home/jovyan/shared/donut/Alzheimer/OAS1/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = (
        f"/home/jovyan/shared/donut/Alzheimer/OAS1/{subject_id}"
        "/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    )
    files = glob.glob(pattern)
    if len(files) == 0:
        return None
    return files[0]

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
print(df.shape)


(234, 14)


In [7]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=1334)
print("train", len(train_df), "val", len(val_df))

train 187 val 47


In [8]:
def load_vol(path):
    img = nib.load(path)
    v = img.get_fdata()
    if v.ndim == 4:
        v = v[:, :, :, 0]
    return v

def cut_empty_slices(v):
    nz = np.argwhere(v > 0)
    lo = nz.min(axis=0)
    hi = nz.max(axis=0)
    return v[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]

def resize_vol(v, shape):
    factors = [t / s for t, s in zip(shape, v.shape)]
    return zoom(v, factors, order=1)

def normalize(v):
    lo, hi = np.percentile(v, [1, 99])
    v = np.clip(v, lo, hi)
    return (v - lo) / (hi - lo + 1e-8)

In [9]:
def preload(frame, target_shape):
    xs, ys = [], []
    paths = frame["path"].tolist()
    labels = frame["label"].tolist()
    for i, (p, lb) in enumerate(zip(paths, labels)):
        v = load_vol(p)
        v = cut_empty_slices(v)
        v = normalize(v)
        v = resize_vol(v, target_shape)
        xs.append(v)
        ys.append(lb)
        if (i + 1) % 20 == 0 or (i + 1) == len(paths):
            print("  โหลดแล้ว", i + 1, "/", len(paths))
    x = torch.tensor(np.stack(xs), dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(ys, dtype=torch.long)
    return x, y

In [10]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix


def run_one_seed(target_shape, seed, x_train, y_train, x_val, y_val, epochs, batch_size):
    set_seed(seed)

    from torch.utils.data import TensorDataset
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size)

    model = resnet18(spatial_dims=3, n_input_channels=1, num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    epoch_times = []
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss = 0

        pbar = tqdm(train_loader, desc=f"shape={target_shape} epoch {epoch+1}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        epoch_times.append(time.time() - t0)

    train_loss_final = running_loss / len(train_loader)
    vram_gb = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0

    # ---- evaluate บน val set เก็บครบทุกค่า ----
    model.eval()
    running_val_loss = 0
    probs, preds, trues = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            running_val_loss += loss.item()
            p = torch.softmax(out, 1)[:, 1].cpu().numpy()
            probs.extend(p)
            preds.extend((p > 0.5).astype(int))
            trues.extend(y.cpu().numpy())

    val_loss_final = running_val_loss / len(val_loader)

    auc = roc_auc_score(trues, probs)
    acc = accuracy_score(trues, preds)
    f1 = f1_score(trues, preds)
    tn, fp, fn, tp = confusion_matrix(trues, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    overfit_gap = val_loss_final - train_loss_final

    return {
        "auc": auc,
        "accuracy": acc,
        "f1": f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "overfit_gap": overfit_gap,
        "vram_gb": vram_gb,
        "avg_epoch_time": avg_epoch_time,
    }


def run_experiment(target_shape, epochs=50, batch_size=2, seeds=(1334,)):
    n_voxels = target_shape[0] * target_shape[1] * target_shape[2]

    print(" preload train...")
    x_train, y_train = preload(train_df, target_shape)
    print(" preload val...")
    x_val, y_val = preload(val_df, target_shape)

    all_results = []
    for seed in seeds:
        print(" seed", seed)
        r = run_one_seed(target_shape, seed, x_train, y_train, x_val, y_val, epochs, batch_size)
        print("  auc", round(r["auc"], 4), " f1", round(r["f1"], 4),
              " sens", round(r["sensitivity"], 4), " spec", round(r["specificity"], 4))
        all_results.append(r)

    def mean_std(key):
        vals = [r[key] for r in all_results]
        return np.mean(vals), np.std(vals)

    auc_mean, auc_std = mean_std("auc")
    acc_mean, acc_std = mean_std("accuracy")
    f1_mean, f1_std = mean_std("f1")
    sens_mean, sens_std = mean_std("sensitivity")
    spec_mean, spec_std = mean_std("specificity")

    return {
        "shape": target_shape,
        "n_voxels": n_voxels,
        "auc_mean": auc_mean, "auc_std": auc_std,
        "acc_mean": acc_mean, "acc_std": acc_std,
        "f1_mean": f1_mean, "f1_std": f1_std,
        "sensitivity_mean": sens_mean, "sensitivity_std": sens_std,
        "specificity_mean": spec_mean, "specificity_std": spec_std,
        "overfit_gap_mean": np.mean([r["overfit_gap"] for r in all_results]),
        "vram_gb": round(max(r["vram_gb"] for r in all_results), 2),
        "sec_per_epoch": round(np.mean([r["avg_epoch_time"] for r in all_results]), 1),
        "n_seeds": len(seeds),
    }

In [12]:
phase1_configs = [(128,128,32), (128,128,64), (128,128,96), (128,128,128)]
phase1_results = []

for shape in phase1_configs:
    print("กำลังทดลอง", shape)
    try:
        r = run_experiment(shape)
        phase1_results.append(r)
        print("  ", r)
    except RuntimeError as e:
        print("  ทำไม่ได้ (อาจ OOM):", e)

กำลังทดลอง (128, 128, 32)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.7389  f1 0.6667  sens 0.6  spec 0.8519
   {'shape': (128, 128, 32), 'n_voxels': 524288, 'auc_mean': np.float64(0.7388888888888889), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.7446808510638298), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6666666666666666), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.6), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.8518518518518519), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.2598850206303376), 'vram_gb': 1.68, 'sec_per_epoch': np.float64(7.4), 'n_seeds': 1}
กำลังทดลอง (128, 128, 64)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.7741  f1 0.6154  sens 0.6  spec 0.7407
   {'shape': (128, 128, 64), 'n_voxels': 1048576, 'auc_mean': np.float64(0.774074074074074), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.6808510638297872), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6153846153846154), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.6), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.7407407407407407), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.1488284455296984), 'vram_gb': 2.76, 'sec_per_epoch': np.float64(13.5), 'n_seeds': 1}
กำลังทดลอง (128, 128, 96)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.8204  f1 0.7273  sens 0.8  spec 0.7037
   {'shape': (128, 128, 96), 'n_voxels': 1572864, 'auc_mean': np.float64(0.8203703703703703), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.7446808510638298), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.7272727272727273), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.8), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.7037037037037037), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(0.8341099486235489), 'vram_gb': 3.84, 'sec_per_epoch': np.float64(19.6), 'n_seeds': 1}
กำลังทดลอง (128, 128, 128)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.7519  f1 0.6818  sens 0.75  spec 0.6667
   {'shape': (128, 128, 128), 'n_voxels': 2097152, 'auc_mean': np.float64(0.7518518518518519), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.7021276595744681), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6818181818181818), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.75), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.6666666666666666), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.2336308074633788), 'vram_gb': 4.92, 'sec_per_epoch': np.float64(26.0), 'n_seeds': 1}


In [13]:
print(f"{'shape':18s} {'voxels':>10s} {'AUC':>16s} {'F1':>16s} {'Sens':>8s} {'Spec':>8s} {'Acc':>7s} {'VRAM(GB)':>9s} {'sec/epoch':>10s}")
for r in phase1_results:
    auc_str = f"{r['auc_mean']:.4f}+-{r['auc_std']:.4f}"
    f1_str = f"{r['f1_mean']:.4f}+-{r['f1_std']:.4f}"
    print(f"{str(r['shape']):18s} {r['n_voxels']:>10,d} {auc_str:>16s} {f1_str:>16s} "
          f"{r['sensitivity_mean']:>8.4f} {r['specificity_mean']:>8.4f} "
          f"{r['acc_mean']:>7.4f} {r['vram_gb']:>9.2f} {r['sec_per_epoch']:>10.1f}")

BEST_Z = max(phase1_results, key=lambda r: r["auc_mean"])["shape"][2]
print()
print("Z ที่ดีที่สุดจากเฟส 1:", BEST_Z)


shape                  voxels              AUC               F1     Sens     Spec     Acc  VRAM(GB)  sec/epoch
(128, 128, 32)        524,288   0.7389+-0.0000   0.6667+-0.0000   0.6000   0.8519  0.7447      1.68        7.4
(128, 128, 64)      1,048,576   0.7741+-0.0000   0.6154+-0.0000   0.6000   0.7407  0.6809      2.76       13.5
(128, 128, 96)      1,572,864   0.8204+-0.0000   0.7273+-0.0000   0.8000   0.7037  0.7447      3.84       19.6
(128, 128, 128)     2,097,152   0.7519+-0.0000   0.6818+-0.0000   0.7500   0.6667  0.7021      4.92       26.0

Z ที่ดีที่สุดจากเฟส 1: 96


In [14]:
print("ใช้ BEST_Z จากเฟส 1:", BEST_Z)

phase2_configs = [(64,64,BEST_Z), (96,96,BEST_Z), (128,128,BEST_Z), (160,160,BEST_Z)]
phase2_results = []

for shape in phase2_configs:
    print("กำลังทดลอง", shape)
    try:
        r = run_experiment(shape)
        phase2_results.append(r)
        print("  ", r)
    except RuntimeError as e:
        print("  ทำไม่ได้ (อาจ OOM):", e)

ใช้ BEST_Z จากเฟส 1: 96
กำลังทดลอง (64, 64, 96)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.763  f1 0.6  sens 0.6  spec 0.7037
   {'shape': (64, 64, 96), 'n_voxels': 393216, 'auc_mean': np.float64(0.7629629629629628), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.6595744680851063), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.6), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.7037037037037037), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.3172599392696884), 'vram_gb': 1.43, 'sec_per_epoch': np.float64(5.8), 'n_seeds': 1}
กำลังทดลอง (96, 96, 96)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.7852  f1 0.6341  sens 0.65  spec 0.7037
   {'shape': (96, 96, 96), 'n_voxels': 884736, 'auc_mean': np.float64(0.7851851851851852), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.6808510638297872), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6341463414634146), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.65), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.7037037037037037), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.1563306045429387), 'vram_gb': 2.42, 'sec_per_epoch': np.float64(11.5), 'n_seeds': 1}
กำลังทดลอง (128, 128, 96)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.7704  f1 0.7317  sens 0.75  spec 0.7778
   {'shape': (128, 128, 96), 'n_voxels': 1572864, 'auc_mean': np.float64(0.7703703703703704), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.7659574468085106), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.7317073170731707), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.75), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.7777777777777778), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(1.1061125551179998), 'vram_gb': 3.84, 'sec_per_epoch': np.float64(19.6), 'n_seeds': 1}
กำลังทดลอง (160, 160, 96)
 preload train...
  โหลดแล้ว 20 / 187
  โหลดแล้ว 40 / 187
  โหลดแล้ว 60 / 187
  โหลดแล้ว 80 / 187
  โหลดแล้ว 100 / 187
  โหลดแล้ว 120 / 187
  โหลดแล้ว 140 / 187
  โหลดแล้ว 160 / 187
  โหลดแล้ว 180 / 187
  โหลดแล้ว 187 / 187
 preload val...
  โหลดแล้ว 20 / 47
  โหลดแล้ว 40 / 47
  โหลดแล้ว 47 / 47
 seed 1334


  auc 0.8426  f1 0.6667  sens 0.75  spec 0.6296
   {'shape': (160, 160, 96), 'n_voxels': 2457600, 'auc_mean': np.float64(0.8425925925925926), 'auc_std': np.float64(0.0), 'acc_mean': np.float64(0.6808510638297872), 'acc_std': np.float64(0.0), 'f1_mean': np.float64(0.6666666666666666), 'f1_std': np.float64(0.0), 'sensitivity_mean': np.float64(0.75), 'sensitivity_std': np.float64(0.0), 'specificity_mean': np.float64(0.6296296296296297), 'specificity_std': np.float64(0.0), 'overfit_gap_mean': np.float64(0.7348504676484603), 'vram_gb': 5.66, 'sec_per_epoch': np.float64(30.1), 'n_seeds': 1}


In [15]:
print(f"{'shape':18s} {'voxels':>10s} {'AUC':>16s} {'F1':>16s} {'Sens':>8s} {'Spec':>8s} {'Acc':>7s} {'VRAM(GB)':>9s} {'sec/epoch':>10s}")
for r in phase2_results:
    auc_str = f"{r['auc_mean']:.4f}+-{r['auc_std']:.4f}"
    f1_str = f"{r['f1_mean']:.4f}+-{r['f1_std']:.4f}"
    print(f"{str(r['shape']):18s} {r['n_voxels']:>10,d} {auc_str:>16s} {f1_str:>16s} "
          f"{r['sensitivity_mean']:>8.4f} {r['specificity_mean']:>8.4f} "
          f"{r['acc_mean']:>7.4f} {r['vram_gb']:>9.2f} {r['sec_per_epoch']:>10.1f}")


shape                  voxels              AUC               F1     Sens     Spec     Acc  VRAM(GB)  sec/epoch
(64, 64, 96)          393,216   0.7630+-0.0000   0.6000+-0.0000   0.6000   0.7037  0.6596      1.43        5.8
(96, 96, 96)          884,736   0.7852+-0.0000   0.6341+-0.0000   0.6500   0.7037  0.6809      2.42       11.5
(128, 128, 96)      1,572,864   0.7704+-0.0000   0.7317+-0.0000   0.7500   0.7778  0.7660      3.84       19.6
(160, 160, 96)      2,457,600   0.8426+-0.0000   0.6667+-0.0000   0.7500   0.6296  0.6809      5.66       30.1
